# Phase 4c: Latest SFT + KV Cache (`6,002` Val Benchmark & Fast Local Chat)

Benchmarks the latest **CoT SFT checkpoint (`e57bf52b...`)** accelerated with **KV caching (`use_cache=True`)** and **equal-length batched greedy decoding (`generate_batch_greedy`)**:

1. Verifies exact token-for-token output parity and measures speedup across Uncached vs KV-Cached Sequential vs Batched KV-Cache.
2. Evaluates **all `6,002` held-out `val` problems** locally with full per-operation & per-category tables (`sealed_test` remains locked until Phase 6).
3. Provides instant KV-cached `ask_math()` & interactive `chat_math()` testing.

In [1]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "main.py").is_file()), Path.cwd().parent)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.infra.eval_setup import setup_notebook_environment, configure_eval_device

project_root = setup_notebook_environment(root)
device, dtype = configure_eval_device("auto")  # 'auto', 'cuda', or 'cpu'

c:\Users\sampa\Dev\MiniReasoner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root located at: C:\Users\sampa\Dev\MiniReasoner
ACTIVE EXECUTION DEVICE: CPU
CPU Threads:        4
Inference Dtype:    torch.float32
Mode:               Portable CPU Inference


## 1. Load Latest CoT SFT Checkpoint

In [2]:
from src.infra.eval_setup import select_run_and_checkpoint, load_checkpoint_bundle

RUN_ID = None                # Auto-selects newest active CoT/scratchpad SFT run (e.g. e57bf52b...)
CHECKPOINT_ALIAS = "latest"  # 'latest' (step 10,000), 'best', or 'step-N'
LOCAL_CHECKPOINT = None

run_id, ckpt_path, snapshot = select_run_and_checkpoint(
    run_id=RUN_ID,
    checkpoint_alias=CHECKPOINT_ALIAS,
    local_checkpoint=LOCAL_CHECKPOINT,
    preferred_stages=("sft",),
    preferred_arith_format="scratchpad",
)
bundle = load_checkpoint_bundle(ckpt_path, device, dtype, snapshot=snapshot)
model, tokenizer, model_config = bundle["model"], bundle["tokenizer"], bundle["model_config"]
arith_manifest = bundle["arith_manifest"]

Discovered 8 run(s) in Azure container 'minireasoner':

Run ID                              Stage     Format      MLflow State  Snapshots  Last Updated (UTC)
--------------------------------------------------------------------------------------------------------
e57bf52bb95842dba6628092261584c4    sft       scratchpad  active        20         2026-11-26 05:56:23
ef3cc27cede44edf90ff5f205a2f71d6    sft       direct      active        16         2026-11-25 17:43:41
e8996fa3b67f48c8951cfd20f68ab40b    sft       direct      active        6          2026-11-25 17:17:13
23a72f3aa8d6487ca401b92b22ae229c    pretrain  -           active        53         2026-11-24 19:09:03
655ddf770790468d988bbb0375fd5aa4    pretrain  -           active        48         2026-11-24 09:22:26
6ef108a339da46679718b67143c39e2e    sft       scratchpad  deleted       4          2026-11-26 05:15:53
f9a6d03c46af4a378895622a82ace4c3    sft       direct      deleted       6          2026-11-25 17:12:04
c26cf6e5ffc749e6

## 2. KV-Cache Exact Parity & Speedup Micro-Benchmark

In [3]:
from src.core.benchmark import benchmark_kv_cache_parity

parity_prompts = [
    "What is -95 * 672?\n",
    "What is 177 * 515?\n",
    "What is 624 / 16?\n",
    "What is -494 - (-377)?\n",
    "What is (10 + 10) + (2 + 3)?\n",
    "If you compute 10 + 10 and also take 2 + 3, what is the sum of both results?\n",
    "Start with 100, subtract 35, and then multiply by 4. What is the final result?\n",
    "Compute 15 * 4 and 92 - 17, then subtract the second result from the first.\n",
]

parity_stats = benchmark_kv_cache_parity(
    model,
    tokenizer,
    parity_prompts,
    device=device,
    max_new_tokens=160,
    batch_size=8,
)
assert parity_stats["seq_parity"] and parity_stats["batch_parity"], "Expected exact token parity with KV cache!" 

KV-CACHE PARITY & SPEEDUP (8 prompts | 499 output tokens | CPU)
Uncached Sequential (`use_cache=False`):  34.06s |   14.7 tok/s | Baseline
KV-Cached Sequential (`use_cache=True`):   9.66s |   51.6 tok/s |  3.52x | parity=True
KV-Cached Batched (bs=8 ):               7.56s |   66.0 tok/s |  4.51x | parity=True


## 3. Full `6,002`-Row Held-Out `val` Benchmark (`generate_batch_greedy`)

In [4]:
from src.core.benchmark import evaluate_arithmetic_split

EVAL_SPLIT = "val"         # All 6,002 held-out val rows (keep 'test' reserved for Phase 6)
MAX_EXAMPLES = None        # Set None for all 6,002 rows, or an int like 500 for a quick subset
BATCH_SIZE = 64 if device.type == "cuda" else 32

full_val_res = evaluate_arithmetic_split(
    model,
    tokenizer,
    model_config,
    arith_manifest,
    device=device,
    dtype=dtype,
    eval_split=EVAL_SPLIT,
    max_examples=MAX_EXAMPLES,
    batch_size=BATCH_SIZE,
    use_kv_cache=True,
    max_misses_to_show=15,
)

PHASE 4 ARITHMETIC EVALUATION | split=val (6,002 rows) | format=scratchpad | multistep=True | kv_cache=True
Dataset:  C:\Users\sampa\Dev\MiniReasoner\data\sft\arithmetic\arith_v3_scratchpad_multistep_add-sub-mul-div_d3_signed_n12000_s42
Hardware: CPU | batch_size=32 | max_new_tokens=163
Computed masked val arith loss: 0.0022 (295,432 targets in 430.4s)
  Decoded 602/6,002 (10.0%) | 35.6s | 16.9 ex/s | ETA 319.3s
  Decoded 1,210/6,002 (20.2%) | 112.0s | 10.8 ex/s | ETA 443.4s
  Decoded 1,819/6,002 (30.3%) | 168.4s | 10.8 ex/s | ETA 387.4s
  Decoded 2,427/6,002 (40.4%) | 260.3s | 9.3 ex/s | ETA 383.4s
  Decoded 3,012/6,002 (50.2%) | 313.9s | 9.6 ex/s | ETA 311.6s
  Decoded 3,620/6,002 (60.3%) | 410.5s | 8.8 ex/s | ETA 270.1s
  Decoded 4,207/6,002 (70.1%) | 532.4s | 7.9 ex/s | ETA 227.2s
  Decoded 4,810/6,002 (80.1%) | 649.7s | 7.4 ex/s | ETA 161.0s
  Decoded 5,424/6,002 (90.4%) | 880.6s | 6.2 ex/s | ETA 93.8s
  Decoded 6,001/6,002 (100.0%) | 1287.2s | 4.7 ex/s | ETA 0.2s
  Decoded 6,002/

## 4. Fast KV-Cached Math Stress Test & Interactive Chat (`ask_math` / `chat_math`)

In [5]:
from src.core.benchmark import create_ask_helper, create_math_chat_helpers

ask_math, chat_math = create_math_chat_helpers(
    model,
    tokenizer,
    device=device,
    answer_format=arith_manifest.get("answer_format", "scratchpad"),
    use_cache=True,
)
ask = create_ask_helper(
    model,
    tokenizer,
    device=device,
    answer_format=arith_manifest.get("answer_format", "scratchpad"),
    use_cache=True,
)

stress_tests = [
    ("What is 177 * 515?", 91155),
    ("What is -95 * 672?", -63840),
    ("What is -494 - (-377)?", -117),
    ("What is 624 / 16?", 39),
    ("What is (10 + 10) + (2 + 3)?", 25),
    ("What is 8 + 8 * 3?", 32),
    ("If you compute 10 + 10 and also take 2 + 3, what is the sum of both results?", 25),
    ("Start with 100, subtract 35, and then multiply by 4. What is the final result?", 260),
    ("Can you do the math for 84 * 12?", 1008),
    ("Compute 15 * 4 and 92 - 17, then subtract the second result from the first.", -15),
]

hits = sum(int(bool(ask_math(q, expected=gold)["correct"])) for q, gold in stress_tests)
print(f"Stress Battery Score: {hits}/{len(stress_tests)} ({hits / len(stress_tests) * 100:.1f}%)")

Q: What is 177 * 515? [MISS | expected 91155]
   Work: 515*100=51500; 515*70=35050; 515*7=3505; 51500+35050=86550; 86550+3505=90055
   Answer: 90055
   -> Extracted: 90055 (94 tok | 2563ms | 36.7 tok/s)
------------------------------------------------------------------------------
Q: What is -95 * 672? [PASS]
   Work: 672*90=60480; 672*5=3360; 60480+3360=63840; neg=-63840
   Answer: -63840
   -> Extracted: -63840 (71 tok | 1984ms | 35.8 tok/s)
------------------------------------------------------------------------------
Q: What is -494 - (-377)? [PASS]
   Work: 494-300=194; 194-70=124; 124-7=117; neg=-117
   Answer: -117
   -> Extracted: -117 (59 tok | 1579ms | 37.4 tok/s)
------------------------------------------------------------------------------
Q: What is 624 / 16? [PASS]
   Work: 16*30=480; 624-480=144; 16*9=144; 30+9=39
   Answer: 39
   -> Extracted: 39 (56 tok | 1656ms | 33.8 tok/s)
------------------------------------------------------------------------------
Q: What is (10 

In [6]:
# Edit CUSTOM_QUESTION to any story or math prompt and re-run:
CUSTOM_QUESTION = "If you compute 45 + 18 and also take 12 * 3, what is the sum of both results?"
_ = ask(CUSTOM_QUESTION)

# Set True (or call chat_math() directly) for multi-turn terminal input
RUN_INTERACTIVE_CHAT = False
if RUN_INTERACTIVE_CHAT:
    chat_math()

Q: If you compute 45 + 18 and also take 12 * 3, what is the sum of both results?
   Work: 45+10=55; 55+8=63; 3*10=30; 3*2=6; 30+6=36; 63+30=93; 93+6=99
   Answer: 99
   -> Extracted: 99 (76 tok | 2195ms | 34.6 tok/s)
------------------------------------------------------------------------------
